# IPM - Interviewer Performance Monitoring (ipmR) on Microsoft Fabric

Runs the R port of the MICS IPM application on fieldwork datasets stored in the attached Lakehouse.

**Lakehouse layout (Files):**
```
Files/ipm/ipmR_0.2.0.tar.gz          <- package (upload once)
Files/ipm/<country>/config.yml        <- settings saved from the Shiny app (or mics_autoconfig)
Files/ipm/<country>/data/*.sav        <- latest fieldwork export (hh, hl, wm, bh, ch, fs, mn ...)
Files/ipm/<country>/results/          <- report, flag tables, anthro listing, CSVs (written by this notebook)
```
R cells use the `%%sparkr` magic (Fabric's R kernel). The analysis itself is plain R (data.table / ggplot2 / officer) and runs on the driver; Spark is only used in the last cell to publish tidy tables to Delta for Power BI. Note: SparkR is deprecated in Spark 4.x (Fabric Runtime 2.0); the final Delta step is written in PySpark so it does not depend on SparkR.

In [ ]:
# Parameters (mark this cell as a parameter cell to drive it from a pipeline)
country = "mycountry"
auto_config = False          # True: build settings automatically from the .sav files (MICS7 conventions)
period_start = ""            # optional override, YYYY-MM-DD (default: value in config.yml)
publish_delta = True

In [ ]:
%%sparkr
# Install dependencies (once per session; or attach them to a Fabric Environment)
need <- c("haven", "data.table", "ggplot2", "officer", "flextable", "openxlsx", "yaml", "ragg")
miss <- need[!vapply(need, requireNamespace, logical(1), quietly = TRUE)]
if (length(miss)) install.packages(miss)
if (!requireNamespace("ipmR", quietly = TRUE))
  install.packages("/lakehouse/default/Files/ipm/ipmR_0.2.0.tar.gz", repos = NULL, type = "source")
library(ipmR)

In [ ]:
%%sparkr
country <- "mycountry"   # keep in sync with the parameter cell
root <- file.path("/lakehouse/default/Files/ipm", country)
cfg_path <- file.path(root, "config.yml")
if (!file.exists(cfg_path)) {
  cfg <- mics_autoconfig(file.path(root, "data"), survey_name = paste("MICS7", country))
  write_config(cfg, cfg_path)
} else cfg <- read_config(cfg_path)
cfg$data_dir    <- file.path(root, "data")
cfg$results_dir <- file.path(root, "results")
print(validate_config(cfg))

In [ ]:
%%sparkr
res <- run_ipm(cfg)   # writes IPM_report.docx, flag_results/*.docx, anthro_report_*.xlsx and data/*.csv
print(res$outputs$flaganal$scores[level %in% c("all", "dataset")])
print(head(res$outputs$flaganal$highly, 20))

In [ ]:
# Publish tidy outputs to Delta tables (PySpark) for a Power BI monitoring dashboard
import os, glob
from pyspark.sql import functions as F
if publish_delta:
    ddir = f"/lakehouse/default/Files/ipm/{country}/results/data"
    for f in glob.glob(os.path.join(ddir, "*.csv")):
        name = os.path.splitext(os.path.basename(f))[0]
        df = (spark.read.option("header", True).option("inferSchema", True)
              .csv(f"Files/ipm/{country}/results/data/{name}.csv")
              .withColumn("country", F.lit(country))
              .withColumn("run_date", F.current_date()))
        df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"ipm_{country}_{name}")
        print("wrote", f"ipm_{country}_{name}")